# IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Dataset:** 590K+ transactions from Vesta Corporation's e-commerce platform  
**Goal:** Understand data structure, class imbalance, missing values, and key fraud signals  
**Stack:** pandas, matplotlib, seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
train_tx = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')

# Merge on TransactionID (left join — not all transactions have identity records)
df = train_tx.merge(train_id, on='TransactionID', how='left')

print(f'Transaction table:  {train_tx.shape[0]:,} rows x {train_tx.shape[1]} cols')
print(f'Identity table:     {train_id.shape[0]:,} rows x {train_id.shape[1]} cols')
print(f'Merged:             {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'\nIdentity match rate: {train_id.shape[0]/train_tx.shape[0]:.1%} of transactions have identity data')

## 2. Dataset Overview

In [ ]:
# Memory usage
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memory usage: {mem_mb:.1f} MB')

# Data types breakdown
dtype_counts = df.dtypes.value_counts()
print('\nColumn dtype breakdown:')
print(dtype_counts.to_string())

# Feature groups (by naming convention)
groups = {
    'Transaction meta': ['TransactionID','TransactionDT','TransactionAmt','ProductCD'],
    'Card features':    [c for c in df.columns if c.startswith('card')],
    'Address/Distance': [c for c in df.columns if c.startswith('addr') or c.startswith('dist')],
    'Email domains':    [c for c in df.columns if 'email' in c.lower()],
    'C-features':       [c for c in df.columns if c.startswith('C') and not c.startswith('card')],
    'D-features':       [c for c in df.columns if c.startswith('D') and c != 'DeviceType' and c != 'DeviceInfo'],
    'M-features':       [c for c in df.columns if c.startswith('M')],
    'V-features':       [c for c in df.columns if c.startswith('V')],
    'Identity (id_)':   [c for c in df.columns if c.startswith('id_')],
    'Device':           [c for c in df.columns if 'evice' in c],
}

print('\nFeature groups:')
for g, cols in groups.items():
    print(f'  {g:<22} {len(cols):>3} columns')

In [ ]:
# --- Class imbalance ---
fraud_rate = df['isFraud'].mean()
fraud_count = df['isFraud'].sum()
legit_count = len(df) - fraud_count

print(f'Fraud transactions:      {fraud_count:>8,.0f}  ({fraud_rate:.2%})')
print(f'Legitimate transactions: {legit_count:>8,.0f}  ({1-fraud_rate:.2%})')
print(f'Class ratio (legit:fraud): {legit_count/fraud_count:.0f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Legitimate', 'Fraud'], [legit_count, fraud_count],
              color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Transactions')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
for bar, val in zip(bars, [legit_count, fraud_count]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()
print(f'\nKey insight: Dataset is heavily imbalanced - fraud is only {fraud_rate:.1%} of transactions.')
print('Model must use class weighting or resampling. Never optimize for accuracy alone.')

## 3. Missing Values Analysis

In [ ]:
# Null percentage per column
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

print(f'Columns with any nulls: {len(null_pct)} / {df.shape[1]}')
print(f'Columns >90% null:      {(null_pct > 90).sum()}')
print(f'Columns >50% null:      {(null_pct > 50).sum()}')
print(f'Columns >20% null:      {(null_pct > 20).sum()}')
print(f'\nTop 20 worst columns:')
print(null_pct.head(20).to_string())

In [ ]:
# Bar chart: null % by feature group
group_null = {}
for g, cols in groups.items():
    valid_cols = [c for c in cols if c in df.columns]
    if valid_cols:
        group_null[g] = df[valid_cols].isnull().mean().mean() * 100

group_null_s = pd.Series(group_null).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d62728' if v > 50 else '#ff7f0e' if v > 20 else '#2ca02c' for v in group_null_s]
bars = ax.barh(group_null_s.index, group_null_s.values, color=colors, edgecolor='white')
ax.axvline(50, color='red', linestyle='--', linewidth=1, alpha=0.6, label='50% threshold')
ax.axvline(90, color='darkred', linestyle='--', linewidth=1, alpha=0.6, label='90% threshold')
ax.set_xlabel('Average Null % within group')
ax.set_title('Missing Values by Feature Group', fontweight='bold')
ax.legend()
for bar, val in zip(bars, group_null_s.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 worst individual columns
top20_null = null_pct.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#d62728' if v > 90 else '#ff7f0e' if v > 50 else '#2ca02c' for v in top20_null.values]
ax.barh(top20_null.index[::-1], top20_null.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(90, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Drop threshold (90%)')
ax.set_xlabel('Null Percentage (%)')
ax.set_title('Top 20 Columns by Missing Value %', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

cols_to_drop = null_pct[null_pct > 90].index.tolist()
print(f'\nRecommendation: Drop {len(cols_to_drop)} columns with >90% nulls')
print(f'Remaining columns after drop: {df.shape[1] - len(cols_to_drop)}')

## 4. Transaction Amount Analysis

In [ ]:
fraud_amt = df[df['isFraud'] == 1]['TransactionAmt']
legit_amt = df[df['isFraud'] == 0]['TransactionAmt']

print('Transaction Amount Statistics:')
print(df.groupby('isFraud')['TransactionAmt'].describe().T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram
ax = axes[0]
ax.hist(legit_amt, bins=100, alpha=0.6, label='Legitimate', color='#4C72B0', density=True)
ax.hist(fraud_amt,  bins=100, alpha=0.7, label='Fraud',      color='#DD8452', density=True)
ax.set_xscale('log')
ax.set_xlabel('Transaction Amount (log scale)')
ax.set_ylabel('Density')
ax.set_title('Amount Distribution: Fraud vs Legit', fontweight='bold')
ax.legend()

# Boxplot by ProductCD
ax = axes[1]
product_order = df.groupby('ProductCD')['TransactionAmt'].median().sort_values(ascending=False).index
df.boxplot(column='TransactionAmt', by='ProductCD', ax=ax,
           order=product_order, sym='', whis=1.5)
ax.set_yscale('log')
ax.set_xlabel('Product Code')
ax.set_ylabel('Transaction Amount (log scale)')
ax.set_title('Amount by Product Code', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'\nMedian fraud amount:      ${fraud_amt.median():.2f}')
print(f'Median legitimate amount: ${legit_amt.median():.2f}')
print(f'Fraud transactions have {fraud_amt.median()/legit_amt.median():.1f}x higher median amounts')

## 5. Temporal Analysis

In [ ]:
# TransactionDT is seconds elapsed since a reference point
# Reference date reverse-engineered from competition: 2017-11-30
BASE_DATE = pd.Timestamp('2017-11-30')
df['dt'] = BASE_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['date']        = df['dt'].dt.date
df['hour_of_day'] = df['dt'].dt.hour
df['day_of_week'] = df['dt'].dt.dayofweek   # 0=Mon, 6=Sun

print(f'Date range: {df["dt"].min().date()} to {df["dt"].max().date()}')
print(f'Span: {(df["dt"].max() - df["dt"].min()).days} days')

In [ ]:
# Daily fraud rate time series
daily = df.groupby('date').agg(
    total=('isFraud','count'),
    fraud=('isFraud','sum')
).reset_index()
daily['fraud_rate'] = daily['fraud'] / daily['total']
daily['date'] = pd.to_datetime(daily['date'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(daily['date'], daily['total'], color='#4C72B0', linewidth=1.2)
axes[0].set_ylabel('Daily Transactions')
axes[0].set_title('Transaction Volume Over Time', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

axes[1].plot(daily['date'], daily['fraud_rate']*100, color='#DD8452', linewidth=1.2)
rolling = daily['fraud_rate'].rolling(7, center=True).mean()
axes[1].plot(daily['date'], rolling*100, color='#d62728', linewidth=2, linestyle='--', label='7-day MA')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xlabel('Date')
axes[1].set_title('Daily Fraud Rate Over Time', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: fraud rate by hour x day-of-week
heatmap_data = df.groupby(['day_of_week', 'hour_of_day'])['isFraud'].mean().unstack() * 100
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd',
            xticklabels=range(24), yticklabels=day_labels,
            cbar_kws={'label': 'Fraud Rate (%)'},
            linewidths=0.5, linecolor='white')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
ax.set_title('Fraud Rate by Hour and Day of Week', fontweight='bold')
plt.tight_layout()
plt.show()

hourly_fraud = df.groupby('hour_of_day')['isFraud'].mean()
peak_hour = hourly_fraud.idxmax()
print(f'Peak fraud hour:   {peak_hour}:00  ({hourly_fraud[peak_hour]:.2%} fraud rate)')
print(f'Lowest fraud hour: {hourly_fraud.idxmin()}:00  ({hourly_fraud.min():.2%} fraud rate)')

## 6. Categorical Feature Analysis

In [ ]:
# Fraud rate by key categoricals
cat_cols = ['ProductCD', 'card4', 'card6']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, cat_cols):
    stats = df.groupby(col)['isFraud'].agg(['mean','count']).reset_index()
    stats = stats.sort_values('mean', ascending=False)
    stats = stats[stats['count'] > 100]
    bars = ax.bar(stats[col].astype(str), stats['mean']*100,
                  color=sns.color_palette('muted', len(stats)),
                  edgecolor='white')
    ax.set_xlabel(col)
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_title(f'Fraud Rate by {col}', fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    for bar, cnt in zip(bars, stats['count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'n={cnt/1e3:.0f}K', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 sender email domains by fraud rate (min 500 transactions)
email_stats = df.groupby('P_emaildomain')['isFraud'].agg(['mean','count'])
email_stats = email_stats[email_stats['count'] >= 500].sort_values('mean', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if v > 0.05 else '#ff7f0e' if v > 0.03 else '#4C72B0'
          for v in email_stats['mean']]
bars = ax.barh(email_stats.index[::-1], email_stats['mean'].values[::-1] * 100,
               color=colors[::-1], edgecolor='white')
ax.axvline(fraud_rate*100, color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({fraud_rate:.1%})')
ax.set_xlabel('Fraud Rate (%)')
ax.set_title('Top 15 Sender Email Domains by Fraud Rate\n(min 500 transactions)', fontweight='bold')
ax.legend()
for bar, (domain, row) in zip(bars, email_stats.iloc[::-1].iterrows()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'n={row["count"]:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Numeric Feature Correlations

In [ ]:
# C-features correlation heatmap
c_cols = [c for c in df.columns if c.startswith('C') and not c.startswith('card')]
c_corr = df[c_cols + ['isFraud']].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(c_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(c_corr, mask=mask, ax=ax, cmap='RdBu_r', center=0,
            vmin=-0.5, vmax=0.5, square=True,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('C-Features Correlation Matrix (incl. isFraud)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 V-features by absolute correlation with isFraud
v_cols = [c for c in df.columns if c.startswith('V')]
v_corr = df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
v_corr_abs = v_corr.abs().sort_values(ascending=False)
top20_v = v_corr_abs.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d62728' if v_corr[i] > 0 else '#4C72B0' for i in top20_v.index]
ax.barh(top20_v.index[::-1], top20_v.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('|Correlation with isFraud|')
ax.set_title('Top 20 V-Features by Absolute Correlation with Fraud', fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 V-features most correlated with fraud:')
for feat in top20_v.head(10).index:
    direction = 'positive' if v_corr[feat] > 0 else 'negative'
    print(f'  {feat}: {v_corr[feat]:+.4f}  ({direction})')

## 8. Key Findings

Summary of the most actionable insights from this EDA:

In [ ]:
# Compute summary stats for findings
product_fraud = df.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False)
highest_product = product_fraud.index[0]
highest_product_rate = product_fraud.iloc[0]

credit_fraud = df[df['card6'] == 'credit']['isFraud'].mean()
debit_fraud  = df[df['card6'] == 'debit']['isFraud'].mean()

identity_fraud    = df[df['DeviceType'].notna()]['isFraud'].mean()
no_identity_fraud = df[df['DeviceType'].isna()]['isFraud'].mean()

print('=' * 65)
print('  KEY FINDINGS -- IEEE-CIS Fraud Detection EDA')
print('=' * 65)
print()
print(f'1. SEVERE CLASS IMBALANCE')
print(f'   Only {fraud_rate:.1%} of {len(df):,} transactions are fraudulent.')
print(f'   Naive accuracy = {1-fraud_rate:.1%}. Use AUC-ROC and precision/recall.')
print()
print(f'2. FRAUD AMOUNTS ARE HIGHER')
print(f'   Median fraud: ${fraud_amt.median():.2f} vs legit: ${legit_amt.median():.2f}')
print(f'   Fraudsters tend to transact at {fraud_amt.median()/legit_amt.median():.1f}x the median.')
print()
print(f'3. PRODUCT TYPE MATTERS')
print(f'   "{highest_product}" has the highest fraud rate at {highest_product_rate:.1%}.')
print(f'   ProductCD is a strong signal -- include as a feature.')
print()
print(f'4. CREDIT CARDS RISKIER THAN DEBIT')
print(f'   Credit fraud rate: {credit_fraud:.1%}  |  Debit fraud rate: {debit_fraud:.1%}')
print(f'   card6 is a useful binary risk signal.')
print()
print(f'5. MISSING IDENTITY = HIGHER FRAUD RISK')
print(f'   Transactions WITHOUT identity data: {no_identity_fraud:.1%} fraud rate')
print(f'   Transactions WITH    identity data: {identity_fraud:.1%} fraud rate')
print(f'   "has_identity" is a strong engineered binary feature.')
print()
print(f'6. DATA QUALITY: HIGH NULL RATE')
print(f'   {len(cols_to_drop)} columns have >90% nulls -- drop before modeling.')
print(f'   V-features are dense (few nulls) and highly predictive.')
print('=' * 65)